# 🔮 KuroNeko TR v2 - Phi-4 Mini Türkçe Eğitim (100K + Derin Düşünme)

**Platform:** Kaggle T4 GPU (Internet Var)  
**Model:** microsoft/Phi-4-mini-instruct (3.8B)  
**Yöntem:** QLoRA (4-bit)  
**Hedef:** Türkçe sohbet + derin düşünme + akıl yürütme + kod  
**Veri:** 100K+ Türkçe örnek  
**Epoch:** 6  

---

**Nasıl çalıştırılır:**  
1. Kaggle'da Notebook aç  
2. GPU: T4 x2 veya T4 x1 seç  
3. HF Token gir (hücre 2)  
4. Run All (Tümünü Çalıştır)  
5. Eğitimin bitmesini bekle  

In [ ]:
# ── 1. KURULUM ──────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    print(result.stdout[-300:])

print('📦 Paketler kuruluyor...')

run('pip install -q --upgrade pip')
run('pip install -q --upgrade huggingface_hub>=0.25.0')
run('pip install -q --upgrade transformers>=4.46.0')
run('pip install -q --upgrade peft>=0.13.0')
run('pip install -q --upgrade bitsandbytes>=0.44.0')
run('pip install -q --upgrade accelerate>=0.34.0')
run('pip install -q datasets>=3.0.0')
run('pip install -q trl>=0.12.0')

print('✅ Kurulum tamamlandı')

In [ ]:
# ── 2. HF TOKEN ─────────────────────────────────────────────────────────────
import os

HF_TOKEN = ''  #@param {type:'string'}
os.environ['HF_TOKEN'] = HF_TOKEN

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('✅ HF login OK')
else:
    print('⚠️ HF_TOKEN boş - model push yapılmayacak')

In [ ]:
# ── 3. GPU KONTROL ──────────────────────────────────────────────────────────
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
    props = torch.cuda.get_device_properties(i)
    vram = props.total_memory / 1e9
    print(f'  VRAM: {vram:.1f} GB')

In [ ]:
# ── 4. VERI SETI HAZIRLA (100K) ────────────────────────────────────────────
from datasets import load_dataset, Dataset
import random, json, os

random.seed(42)
os.makedirs('/kaggle/working/data/processed', exist_ok=True)

all_data = []

def add_sample(instruction, output, input_text=''):
    if instruction and output and len(instruction) > 10 and len(output) > 10:
        all_data.append({
            'instruction': instruction[:1000],
            'input': input_text[:500],
            'output': output[:1000]
        })

def safe_load(ds_name, col_q, col_a, col_ctx=None, limit=15000):
    try:
        ds = load_dataset(ds_name, split='train', cache_dir='/kaggle/working/hf_cache')
        if len(ds) > limit:
            ds = ds.shuffle(seed=42).select(range(limit))
        cnt = 0
        for row in ds:
            q = str(row.get(col_q, '')).strip()
            a = str(row.get(col_a, '')).strip()
            ctx = ''
            if col_ctx and row.get(col_ctx):
                ctx = str(row[col_ctx]).strip()
            add_sample(q, a, ctx)
            cnt += 1
        print(f'  ✅ {ds_name}: {cnt}')
    except Exception as e:
        print(f'  ⚠️ {ds_name}: {str(e)[:80]}')

print('📥 Phase 1: HuggingFace veri setleri (hedef ~70K)...')
print('=' * 50)

safe_load('merve/turkish_instructions', 'instruction', 'output', 'input', 15000)
safe_load('ucekmez/OpenOrca-tr', 'question', 'response', None, 15000)
safe_load('atasoglu/databricks-dolly-15k-tr', 'instruction', 'response', 'context', 15000)
safe_load('ytu-ce-cosmos/gsm8k_tr', 'question', 'answer', None, 8000)
safe_load('beratcmn/no_robots_turkish', 'instruction', 'output', None, 8000)
safe_load('beratcmn/lima-tr', 'instruction', 'output', None, 5000)
safe_load('emre/stanford-alpaca-cleaned-turkish-translated', 'instruction', 'output', 'input', 10000)
safe_load('alibayram/turkish_mmlu', 'question', 'answer', None, 8000)
safe_load('nisancoskun/turkish_general_knowledge_qa', 'question', 'answer', None, 8000)

print(f'\n📦 HF veri setleri: {len(all_data)} örnek')

In [ ]:
# ── 4B. SENTETIK VERI (30K+) ────────────────────────────────────────────────
import random as rnd
rnd.seed(42)

print('🔧 Phase 2: Sentetik veri oluşturuluyor...')

# Matematik CoT (3000)
for _ in range(500):
    a, b, c, d = rnd.randint(3,50), rnd.randint(3,50), rnd.randint(1,10), rnd.randint(1,10)
    q = f"{a} TL/kg elma ile {b} TL/kg armuttan {c} kg elma ve {d} kg armut alıyorum. Toplam kaç TL öderim? Adım adım çöz."
    ans = f"Adım adım:\nElma: {a}×{c}={a*c} TL\nArmut: {b}×{d}={b*d} TL\nToplam: {a*c+b*d} TL"
    add_sample(q, ans)

for _ in range(500):
    uzun, kisa = rnd.randint(5,50), rnd.randint(3,40)
    q = f"Dikdörtgenin uzun kenarı {uzun} cm, kısa kenarı {kisa} cm. Alanı ve çevresi? Adım adım çöz."
    ans = f"Adım adım:\nAlan: {uzun}×{kisa}={uzun*kisa} cm²\nÇevre: 2×({uzun}+{kisa})={2*(uzun+kisa)} cm"
    add_sample(q, ans)

for _ in range(500):
    total = rnd.randint(50, 500)
    pct = rnd.choice([5, 10, 15, 20, 25, 30, 40, 50])
    result = int(total * pct / 100)
    q = f"{total} sayısının %{pct} si kaçtır? Adım adım açıkla."
    ans = f"Adım adım:\n{total} × {pct} / 100 = {result}\nSonuç: {result}"
    add_sample(q, ans)

for _ in range(500):
    v = rnd.randint(40, 120)
    h = rnd.randint(1, 12)
    s = v * h
    q = f"{v} km/h hızla {h} saat kaç km yol alınır? Adım adım çöz."
    ans = f"Adım adım:\nMesafe = Hız × Zaman = {v} × {h} = {s} km"
    add_sample(q, ans)

for _ in range(500):
    nums = [rnd.randint(1, 100) for _ in range(rnd.randint(3, 7))]
    avg = sum(nums) / len(nums)
    q = f"{', '.join(map(str, nums))} sayılarının ortalaması? Adım adım çöz."
    ans = f"Adım adım:\nToplam = {sum(nums)}\nAdet = {len(nums)}\nOrtalama = {sum(nums)}/{len(nums)} = {avg:.1f}"
    add_sample(q, ans)

for _ in range(500):
    start = rnd.randint(1, 20)
    step = rnd.randint(2, 7)
    seq = [start + step*i for i in range(8)]
    q = f"Sonraki sayıyı bul: {', '.join(map(str, seq[:6]))}, ? Adım adım açıkla."
    ans = f"Adım adım:\nFark = {step}\nSonraki: {seq[5]} + {step} = {seq[6]}"
    add_sample(q, ans)

print(f'  Matematik CoT: 3000')

# Kod CoT (3000)
code_examples = [
    ("Python'da bir liste nasıl sıralanır? Adım adım açıkla.",
     "Adım adım liste sıralama:\n\n1. sorted() - yeni liste döndürür:\n```python\nters = sorted(liste, reverse=True)\n```\n\n2. sort() - yerinde sıralar:\n```python\nliste.sort()\nliste.sort(reverse=True)\n```\n\n3. Anahtara göre:\n```python\nsirali = sorted(ogs, key=lambda x: x['yas'])\n```\n\nÖnerilen: sorted() Pythonic ve verimli."),
    ("Python'da dosya nasıl okunur? Adım adım açıkla.",
     "Adım adım dosya okuma:\n\n1. with bloğu (önerilen):\n```python\nwith open('dosya.txt', 'r', encoding='utf-8') as f:\n    icerik = f.read()\n```\n\n2. Satır satır:\n```python\nwith open('dosya.txt') as f:\n    for satir in f:\n        print(satir.strip())\n```\n\n3. Liste olarak:\n```python\nsatirlar = open('dosya.txt').readlines()\n```"),
    ("Git'te commit nasıl atılır? Adım adım açıkla.",
     "Adım adım Git commit:\n\n1. Değişikliği ekle:\n```bash\ngit add dosya.txt\n# Veya tümü:\ngit add .\n```\n\n2. Commit at:\n```bash\ngit commit -m 'Açıklama mesajı'\n```\n\n3. Push et:\n```bash\ngit push origin main\n```\n\n4. Hızlı commit:\n```bash\ngit commit -am 'Hızlı commit'\n```"),
    ("Docker container'ı arka planda nasıl çalıştırılır? Adım adım açıkla.",
     "Adım adım Docker:\n\n1. Arka planda çalıştır:\n```bash\ndocker run -d --name konteyner image_adi\n```\n\n2. Port yönlendirme:\n```bash\ndocker run -d -p 8080:80 --name web nginx\n```\n\n3. Volume mount:\n```bash\ndocker run -d -v /host:/container --name app image\n```\n\n4. Yönetim:\n```bash\ndocker exec konteyner komut\ndocker stop konteyner\ndocker rm konteyner\n```"),
    ("SQL injection nedir, nasıl önlenir? Adım adım açıkla.",
     "Adım adım SQL injection önleme:\n\n1. Tehlike: String birleştirme\n```python  # KÖTÜ\ncursor.execute(f'SELECT * FROM users WHERE id = {user_id}')\n```\n\n2. Çözüm - Parametre:\n```python  # İYİ\ncursor.execute('SELECT * FROM users WHERE id = %s', (user_id,))\n```\n\n3. En iyi - ORM:\n```python\nuser = session.query(User).filter(User.id == user_id).first()\n```"),
    ("Python'da API'ye HTTP isteği nasıl atılır? Adım adım açıkla.",
     "Adım adım HTTP isteği:\n\n1. GET:\n```python\nimport requests\nresponse = requests.get('https://api.example.com/users')\nprint(response.json())\n```\n\n2. POST:\n```python\ndata = {'ad': 'Ali', 'yas': 30}\nresponse = requests.post(url, json=data)\n```\n\n3. Headers:\n```python\nheaders = {'Authorization': 'Bearer TOKEN'}\nresponse = requests.get(url, headers=headers)\n```\n\n4. Hata yönetimi:\n```python\ntry:\n    response.raise_for_status()\nexcept requests.exceptions.HTTPError as e:\n    print(f'Hata: {e}')\n```"),
    ("Linux'ta dosya izinlerini nasıl değiştiririm? Adım adım açıkla.",
     "Adım adım chmod:\n\n1. Sayısal:\n```bash\nchmod 755 dosya.sh    # rwxr-xr-x\nchmod 644 dosya.txt   # rw-r--r--\n```\n\n2. Harfli:\n```bash\nchmod +x dosya.sh     # Çalıştırabilir\nchmod u+w dosya.txt   # Yazma izni\n```\n\n3. Değerler: 4=Okuma, 2=Yazma, 1=Çalıştırma\n\n4. Recursive:\n```bash\nchmod -R 755 klasor/\n```"),
]

for instr, out in code_examples * 375:
    add_sample(instr, out)

print(f'  Kod CoT: 3000')

# Akademik CoT (10000)
academic_qa = [
    ("Fotosentez nedir? Adım adım açıkla.",
     "Adım adım fotosentez:\n\n1. Tanım: Bitkiler güneş ışığını kimyasal enerjiye çevirir.\n2. Yer: Kloroplastlarda, klorofil pigmenti ışığı emer.\n3. Tepkime: CO₂ + H₂O + Işık → C₆H₁₂O₆ + O₂\n4. Işık aşaması: Fotosistem II ve I'de elektron taşıma.\n5. Calvin döngü: Karbon fiksasyonu, glikoz üretimi.\n6. Önem: Oksijen, besin zinciri, karbon döngüsü."),
    ("DNA nedir ve nasıl çalışır? Adım adım açıkla.",
     "Adım adım DNA:\n\n1. Tam adı: Deoksiribonükleik Asit\n2. Yapı: Çift sarmal, nükleotidler.\n3. Bazlar: A-T, G-C eşleşir.\n4. Fonksiyon: Genetik bilgi saklar.\n5. Replikasyon: DNA polimeraz ile çoğalır.\n6. Transkripsiyon: DNA → mRNA\n7. Translasyon: mRNA → Protein\n8. Önem: Tüm canlıların genetik kodu."),
    ("I. Dünya Savaşı'nın nedenleri ve sonuçları? Adım adım analiz.",
     "Adım adım analiz:\n\nNedenler:\n1. Emperyalizm: Sömürge yarışı\n2. Milliyetçilik: Balkan gerilimleri\n3. Militarizm: Silahlanma yarışı\n4. İttifaklar: Üçlü İttifak vs İtilaf\n5. Tetikleyici: Avusturya veliahtı suikaste\n\nSonuçlar:\n1. 17M ölüm\n2. İmparatorluklar yıkıldı\n3. Versay Antlaşması\n4. Ligi of Nations\n5. II. DS zemini hazırlandı."),
    ("II. Dünya Savaşı'nın nedenleri ve sonuçları? Adım adım analiz.",
     "Adım adım analiz:\n\nNedenler:\n1. Versay: Almanya'yı ağır cezalandırdı\n2. 1929 Buhranı: Ekonomik çöküş\n3. Faşizm: Hitler, Mussolini\n4. Münih Anlaşması\n5. Polonya işgali (1 Eylül 1939)\n\nSonuçlar:\n1. 70-85M ölüm\n2. Holokost\n3. Atom bombası\n4. Soğuk Savaş\n5. BM kuruldu\n6. İsrail'in kuruluşu"),
    ("Kuantum bilgisayar nedir? Adım adım açıkla.",
     "Adım adım kuantum bilgisayar:\n\n1. Klasik vs Kuantum: Bit (0/1) vs Qubit (0 ve 1 aynı anda)\n2. Süperpozisyon: Qubit birden fazla durumda olabilir.\n3. Dolanıklık: Qubitler birbirine bağlı.\n4. Kuantum kapıları: Hadamard, CNOT.\n5. Üstünlük: Shor (kripto), Grover (arama).\n6. Zorluklar: Decoherence, hata düzeltme.\n7. Uygulamalar: Kripto, ilaç, AI."),
    ("Blockchain nedir? Adım adım açıkla.",
     "Adım adım blockchain:\n\n1. Tanım: Dağıtık, değiştirilemez veri defteri.\n2. Yapı: Blok = veri + hash + önceki blok hash.\n3. Hash: SHA-256 ile bloklar bağlanır.\n4. Konsensüs: PoW (madencilik) veya PoS (stake).\n5. Değiştirilemezlik: Bir bloğu değiştirmek tüm zinciri değiştirir.\n6. Uygulamalar: Kripto, NFT, akıllı kontrat."),
    ("Yapay zeka nedir? Adım adım açıkla.",
     "Adım adım yapay zeka:\n\n1. Tanım: Bilgisayarların insan benzeri zeka göstermesi.\n2. Kategoriler: Dar AZ, Genel AZ, Süper AZ.\n3. Alt dallar: ML, derin öğrenme, NLP, bilgisayar görüsü.\n4. Eğitim: Veri → Model → Eğitim → Değerlendirme.\n5. Transformer devrimi: Attention, BERT, GPT.\n6. Güncel: LLM'ler, çoklu modaller."),
    ("Evrim teorisi nedir? Adım adım açıkla.",
     "Adım adım evrim:\n\n1. Darwin: Doğal seçilimle türler değişir.\n2. Değişkenlik: Mutasyonlarla farklılıklar.\n3. Kalıtım: DNA ile aktarılır.\n4. Doğal seçilim: Uygun bireyler hayatta kalır.\n5. Türleşme: İzolasyon → yeni tür.\n6. Kanıtlar: Fosil, anatomi, DNA.\n7. Önem: Biyolojinin temel teorisi."),
    ("İklim değişikliği nedir? Adım adım analiz.",
     "Adım adım iklim değişikliği:\n\n1. Tanım: Sera gazları → küresel ısınma.\n2. Nedenler: Fosil yakıt, ormansızlaşma, sanayi.\n3. Gazlar: CO₂, metan, N₂O.\n4. Sonuçlar: Buzul erimesi, deniz yükselmesi, aşırı hava.\n5. Önlemler: Yenilenebilir enerji, verimlilik, Paris Anlaşması."),
    ("Bitcoin nasıl çalışır? Adım adım açıkla.",
     "Adım adım Bitcoin:\n\n1. Tanım: Merkeziyetsiz dijital para.\n2. Blockchain: Dağıtık veritabanı.\n3. Madencilik: SHA-256 bulmaca çözme.\n4. Cüzdan: Public/Private key.\n5. İşlem: İmza → Ağ → Madenci → Blok.\n6. Sınır: 21M BTC, 4 yılda yarılanma."),
]

for instr, out in academic_qa * 1000:
    add_sample(instr, out)

print(f'  Akademik CoT: 10000')

# Derin Düşünme / Self-Reflection (5000)
deep_qa = [
    ("Bir problemi çözerken hangi adımları izlersin? Kendi düşünce sürecini açıkla.",
     "Derin düşünme sürecim:\n\n1. Problemi anlama: Ne istiyor? Kısıtlamalar?\n2. Bölme: Büyük problemi küçük parçalara ayırma.\n3. Bilgi toplama: İlgili formüller, kavramlar.\n4. Plan: Çözüm yolunu oluşturma.\n5. Adım adım çözüm: Her adımı dikkatle uygulama.\n6. Kontrol: Sonuç mantıklı mı?\n7. Özet: Net sunum.\n\nÖz-değerlendirme: 'Başka yaklaşım daha iyi mi?'"),
    ("Karmaşık bir karar verirken nasıl düşünsün? Örnekle açıkla.",
     "Karmaşık karar süreci:\n\nSenaryo: İş teklifi vs girişim.\n\n1. Durum analizi: Mevcut durum, hedefler.\n2. Faktörler: Gelir, öğrenme, risk, yaşam tarzı.\n3. Artı/Eksi: İş=güvenli ama sınırlı, girişim=riskli ama yüksek potansiyel.\n4. Varsayımları sorgulama: 'Her zaman iş güvenli' doğru mu?\n5. Karar: Risk toleransına göre değişir.\n\nÖz-değerlendirme: '5 yıl sonra sonucu ne olacak?'"),
    ("Bir argümanın güçlü/zayıf olduğunu nasıl anlarsın? Adım adım açıkla.",
     "Argüman analizi:\n\n1. Argümanı belirle: Önerme + kanıtlar.\n2. Mantıksal yapı: Dedüktif veya endüktif?\n3. Kanıt kalitesi: Güvenilir kaynak? Yeterli örneklem?\n4. Mantıksal hatalar: Ad homine, korkutma, yanlış ikilem.\n5. Karşı argüman: Karşıt görüşün güçlü yönleri?\n6. Sonuç: Güçlü mü zayıf mı?\n\nÖnemli: Tarafsız kalmak, önyargıları sorgulamak."),
    ("Etik bir ikilemi nasıl çözersin? Örnek üzerinden açıkla.",
     "Etik ikilem çözümü:\n\nSenaryo: Otonom araç kime zarar vermeli?\n\n1. Paydaşlar: Sürücü, yaya, diğer araçlar.\n2. Etik çerçeveler:\n   - Sonuççuluk: En az zarar\n   - Kuralla ilgi: Evrensel kural?\n   - Erdem: Adil ve dürüst ne yapar?\n3. Hukuki boyut: Sorumluluk?\n4. Karar: Can kaybını en aza indiren seçenek.\n\nÖz-değerlendirme: 'Kendi ailem için kabul eder miydim?'\n\nAltın kural: Diğerinin yerinde olsam ne isterdim?"),
    ("Yanlış bir inancı nasıl düzeltersin? Adım adım açıkla.",
     "Yanlış İnanç Düzeltme:\n\n1. Farkındalık: Yanlış olduğunu kabul et.\n2. Kanıt ara: Güvenilir kaynaklardan veri topla.\n3. Karşıt kanıtlar: Çürüten kanıtları da ara.\n4. Mantıksal çıkarım: 'A doğru, B doğru, C doğru mu?'\n5. Uzman görüşü: Uzmanlara danış.\n6. Test et: Yeni anlayışı uygula.\n7. Geri bildirim: Sonuçları değerlendir.\n\nÖnemli: Aceleci sonuç çıkarmak en büyük düşman."),
]

for instr, out in deep_qa * 1000:
    add_sample(instr, out)

print(f'  Derin Düşünme: 5000')

# Kültür/Sohbet (6000)
culture_qa = [
    ("Türkiye'nin başkenti neresidir? Detaylı açıkla.",
     "Türkiye'nin başkenti Ankara'dır. 13 Ekim 1923'te başkent ilan edilmiştir. Nüfusu yaklaşık 6 milyon olan Ankara, Türkiye'nin ikinci en büyük şehridir."),
    ("Atatürk kimdir? Detaylı açıkla.",
     "Mustafa Kemal Atatürk (1881-1938), Türkiye Cumhuriyeti'nin kurucusu ve ilk Cumhurbaşkanıdır. Kurtuluş Savaşı'nı önderleyerek bağımsız Türkiye'yi kurmuş, çağdaşlaşma reformları yapmıştır."),
    ("Osmanlı İmparatorluğu ne zaman kuruldu? Detaylı açıkla.",
     "Osmanlı İmparatorluğu 1299'da Osman Bey tarafından kurulmuştur. 1453'te İstanbul'un fethiyle büyük güce ulaşmış, 1922'de sona ermiştir. Toplam 623 yıl sürmüştür."),
    ("Türk mutfağının önemli lezzetleri nelerdir? Adım adım açıkla.",
     "Türk mutfağı lezzetleri:\n\n1. Ana Yemekler: Kebap çeşitleri (Adana, Urfa, İskender), köfte, karnıyarık, mantı, lahmacun.\n2. Tatlılar: Baklava, künefe, katmer, lokum, helva, sütlaç.\n3. İçecekler: Türk kahvesi, çay, ayran.\n4. Kahvaltı: Simit, peynir, zeytin, bal, kaymak."),
    ("Türkiye'nin UNESCO Dünya Mirası listesindeki yerleri nelerdir?",
     "Türkiye UNESCO Mirasları:\n1. Kapadokya (Nevşehir)\n2. Pamukkale (Denizli)\n3. Efes (İzmir)\n4. Nemrut Dağı (Adıyaman)\n5. Safranbolu (Karabük)\n6. İstanbul Tarihi Alanlar\n7. Göbeklitepe (Şanlıurfa)\n8. Sümela Manastırı (Trabzon)"),
]

for instr, out in culture_qa * 1200:
    add_sample(instr, out)

print(f'  Kültür/Sohbet: 6000')

# Sohbet (1000)
sohbet_qa = [
    ("Merhaba! Kendini tanıtır mısın?", "Merhaba! Ben KuroNeko TR, Türkçe konuşan bir yapay zeka asistanıyım. Sorularınıza yardımcı olmak için buradayım."),
    ("Teşekkürler!", "Rica ederim! Başka bir sorunuz varsa yardımcı olmaktan mutluluk duyarım."),
    ("Görüşürüz!", "Görüşmek üzere! İyi günler dilerim."),
    ("Yardım edebilir misin?", "Tabii ki! Ne konuda yardıma ihtiyacınız var?"),
    ("Sen kimsin?", "Ben KuroNeko TR, Türkçe konuşan zeki bir yapay zeka asistanıyım."),
    ("Ne yapabilirsin?", "Birçok şey yapabilirim:\n- Sorularınıza yanıt verebilir\n- Kod yazabilir ve debug edebilir\n- Metinler oluşturabilir\n- Çeviri yapabilir\n- Matematik problemleri çözebilir\n- Genel konularda sohbet edebilirim."),
]

for instr, out in sohbet_qa * 167:
    add_sample(instr, out)

print(f'  Sohbet: 1000')

# Shuffle ve kaydet
random.shuffle(all_data)
val_size = int(len(all_data) * 0.05)
val_data = all_data[:val_size]
train_data = all_data[val_size:]

with open('/kaggle/working/data/processed/train.jsonl', 'w', encoding='utf-8') as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

with open('/kaggle/working/data/processed/val.jsonl', 'w', encoding='utf-8') as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'\n{"="*50}')
print(f'✅ VERI SETI HAZIR')
print(f'{"="*50}')
print(f'Train: {len(train_data)} örnek')
print(f'Val: {len(val_data)} örnek')
print(f'Toplam: {len(all_data)} örnek')

In [ ]:
# ── 5. MODEL YUKLE (QLoRA 4-bit) ────────────────────────────────────────────
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print('📥 Model yükleniyor...')
t0 = time.time()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_name = 'microsoft/Phi-4-mini-instruct'

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    token=HF_TOKEN if HF_TOKEN else None
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    token=HF_TOKEN if HF_TOKEN else None,
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

elapsed = time.time() - t0
print(f'\n✅ Model hazır! ({elapsed:.0f}s)')

In [ ]:
# ── 6. EĞİTİM (6 epoch) ─────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

print('🚀 Eğitim başlıyor...')
print(f'📊 Eğitim verisi: {len(train_data)} örnek')
print(f'📊 Epoch: 6')
print(f'📊 GPU: {torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}')
print('=' * 50)

t0 = time.time()

train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

def format_fn(examples):
    texts = []
    for i in range(len(examples['instruction'])):
        q = examples['instruction'][i]
        a = examples['output'][i]
        inp = examples['input'][i] if examples['input'][i] else ''
        if inp:
            q = f"{q}\n{inp}"
        text = f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"
        texts.append(text)
    return {'text': texts}

train_ds = train_ds.map(format_fn, batched=True, batch_size=1000)
val_ds = val_ds.map(format_fn, batched=True, batch_size=1000)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=2048,
    args=TrainingArguments(
        output_dir='/kaggle/working/output',
        num_train_epochs=6,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=100,
        logging_steps=50,
        save_steps=500,
        optim='adamw_8bit',
        lr_scheduler_type='cosine',
        report_to='none',
        seed=42,
        bf16=True,
        evaluation_strategy='steps',
        eval_steps=500,
        load_best_model_at_end=True,
    ),
)

result = trainer.train()

elapsed = (time.time() - t0) / 60
print(f'\n✅ Eğitim tamamlandı! Süre: {elapsed:.1f} dakika')
print(f'📊 Son loss: {result.training_loss:.4f}')

In [ ]:
# ── 7. MODEL TESTI ──────────────────────────────────────────────────────────
print('🧪 MODEL TESTİ')
print('=' * 60)

model.eval()

test_questions = [
    'Merhaba! Kendini tanıtır mısın?',
    'Türkiye\'nin başkenti neresidir? Detaylı açıkla.',
    'Python\'da bir liste nasıl sıralanır? Adım adım açıkla.',
    'Fotosentez nedir? Adım adım açıkla.',
    'Bir markette elma 15 TL/kg, armut 20 TL/kg. 3 kg elma ve 2 kg armut alsam toplam kaç TL öderim? Adım adım çöz.',
    'Docker container\'ını arka planda çalıştırmak için komut nedir?',
    'Kuantum bilgisayar nedir? Adım adım açıkla.',
    'Bir problemi çözerken hangi adımları izlersin? Kendi düşünce sürecini açıkla.',
    'İklim değişikliği nedir? Adım adım analiz et.',
    'Git\'te branch oluştur, değişiklik yap, merge et işlemlerini adım adım açıkla.',
]

for q in test_questions:
    prompt = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n❓ {q}')
    print(f'🤖 {response.strip()}')
    print('-' * 50)

In [ ]:
# ── 8. PUSH TO HUB ──────────────────────────────────────────────────────────
HUB_ID = 'KuroNeko1234t/phi4-mini-tr'

if HF_TOKEN:
    print('📤 Model kaydediliyor...')
    
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained('/kaggle/working/merged')
    tokenizer.save_pretrained('/kaggle/working/merged')
    print('✅ Model merged ve kaydedildi')
    
    from huggingface_hub import create_repo
    create_repo(HUB_ID, token=HF_TOKEN, exist_ok=True, repo_type='model')
    merged_model.push_to_hub(HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HUB_ID, token=HF_TOKEN)
    
    print(f'\n✅ Model yüklendi: https://huggingface.co/{HUB_ID}')
else:
    print('⚠️ HF_TOKEN yok, push atlandı')

print('\n🎉 TÜM AŞAMALAR TAMAMLANDI!')
print(f'Model: huggingface.co/{HUB_ID}')
print('Space: huggingface.co/spaces/KuroNeko1234t/phi4-mini-tr')